# 07. The data contract

**The question this notebook answers:** can an engineer who has never seen the raw files build a model from what Part A hands over, without asking us anything, and can anyone check that the hand-over is faithful?

**What the brief asks:** a folder `contract/` with five files, **all transforms fitted only on loans issued up to 31 December 2014**, and **a script that verifies `reference_transforms.csv`**. The model engineer trains on loans up to the cutoff, validates on 2015, uses `transform.joblib` **unchanged** (never refitting it), and proves parity on the 50 reference rows at tolerance 1e-6.

| File | What it holds | Used for |
|---|---|---|
| `analytical_dataset.parquet` | one cleaned row per loan: cleaned fields, features, factor scores, label | training and validation data |
| `transform.joblib` | the fitted pipeline: raw application and bureau record in, cleaned fields, features, WoE, PCA components and factor scores out | the first step of training and of scoring |
| `data_spec.json` | each field's type, valid range, cleaning rule, reliability, the factor definitions and the WoE bins and groups | input validation, and deciding which fields to trust |
| `reference_transforms.csv` | 50 raw rows with the expected transformed values | the parity test |
| `findings.md` | the analysis report: a manager summary, then the technical appendix | model design decisions and the model card |

In [1]:
import os, json, warnings
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")

import joblib
import numpy as np
import pandas as pd

from credit_risk.contract_check import read_reference, verify_contract
from credit_risk.io import FIT_CUTOFF, load_applications, load_holdout, load_servicing
from credit_risk.spec import ContractViolation, validate_raw

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 110)

C = Path("contract")
ds = pd.read_parquet(C / "analytical_dataset.parquet")
ds["issue_d"] = pd.to_datetime(ds.issue_d)
spec = json.load(open(C / "data_spec.json"))
transformer = joblib.load(C / "transform.joblib")
print(pd.Series({p.name: f"{p.stat().st_size / 1024:,.0f} KB" for p in sorted(C.iterdir())}).to_string())

analytical_dataset.parquet    11,046 KB
data_spec.json                    87 KB
findings.md                        9 KB
reference_transforms.csv         109 KB
transform.joblib                  23 KB


## 1. `analytical_dataset.parquet`: the training table

One row per loan, 2013 to 2015. The engineer splits it in time: loans up to the cutoff to train, 2015 to validate.

In [2]:
print(f"{ds.shape[0]:,} rows x {ds.shape[1]} columns")
print(ds.groupby(ds.issue_d.dt.year).defaulted.agg(loans="size", default_rate="mean").round(3).T.to_string())
print(f"\nloans up to the cutoff (training): {(ds.issue_d <= FIT_CUTOFF).sum():,}   2015 (validation): {(ds.issue_d > FIT_CUTOFF).sum():,}")

def group(c):
    if c in ("id", "issue_d", "defaulted"): return "key, date, label"
    if c.endswith("_woe"): return "Weight of Evidence columns"
    if c.startswith("pc_"): return "PCA components"
    if c.startswith("factor_"): return "factor scores"
    if c.endswith(("_fixed", "_invalid", "_imputed", "_recoded", "_is_zero", "_missing")): return "repair and indicator flags"
    if c.endswith(("_log", "_treated", "_capped")): return "treated fields (capped, logged, filled)"
    return "cleaned fields and features"
print("\ncolumns by kind:")
print(pd.Series([group(c) for c in ds.columns]).value_counts().to_string())

30,000 rows x 148 columns
issue_d           2013     2014       2015
loans         5349.000  9024.00  15627.000
default_rate     0.159     0.18      0.201

loans up to the cutoff (training): 14,373   2015 (validation): 15,627

columns by kind:
cleaned fields and features                50
Weight of Evidence columns                 37
treated fields (capped, logged, filled)    31
repair and indicator flags                 12
factor scores                               8
PCA components                              7
key, date, label                            3


In [3]:
leaks = set(load_servicing().columns) - {"id"} | {"bureau_recovery_flag"}
print("servicing columns or the leaked recovery flag present in the dataset:", sorted(leaks & set(ds.columns)) or "none")
print("missing values in the label or the date:", int(ds[["defaulted", "issue_d"]].isna().sum().sum()))

servicing columns or the leaked recovery flag present in the dataset: none
missing values in the label or the date: 0


**Reading it.** The table holds every loan, treated and ready to model, with the label. It contains **no servicing column and not the leaked `bureau_recovery_flag`**, so a model built on it cannot cheat. The kinds of column are separated by name, so an engineer can pick exactly what they want: raw cleaned fields, WoE columns for the scorecard, or PCA and factor scores for the ablation.

## 2. `data_spec.json`: what each field is, and whether to trust it

The brief gives the required shape. Our entries carry those keys, plus the field's role and whether it was known at application time.

In [4]:
common = {"name", "type", "cleaning", "missing_strategy", "reliability", "reliability_reason"}
def required_keys(field):
    # the brief's keys; a range (min, max) for numbers and dates, the allowed values for categories, neither for free text
    if field["type"] in ("float", "int", "date"): return common | {"min", "max"}
    if field["type"] == "category": return common | {"allowed"}
    return common
required_factor = {"name", "method", "top_loadings", "variance_explained"}
print("top-level keys:", list(spec))
print("contract_version:", spec["contract_version"], "| fit_cutoff:", spec["fit_cutoff"])
print("every field has the keys that apply to its type:", all(required_keys(f) <= set(f) for f in spec["fields"]), f"({len(spec['fields'])} fields; types: {pd.Series([f['type'] for f in spec['fields']]).value_counts().to_dict()})")
print("every factor has the required keys:", all(required_factor <= set(f) for f in spec["factors"]), f"({len(spec['factors'])} factors)")
print("\none field, as stored:"); print(json.dumps(next(f for f in spec["fields"] if f["name"] == "annual_inc"), indent=1))
print("\none factor, as stored:"); print(json.dumps(spec["factors"][0], indent=1))

top-level keys: ['contract_version', 'fit_cutoff', 'label', 'notes', 'fields', 'treated_fields', 'category_groupings', 'cleaning_rules', 'woe', 'pca', 'factors', 'excluded_fields', 'platform_score_fields', 'restricted_fields']
contract_version: 1.0.0 | fit_cutoff: 2014-12-31
every field has the keys that apply to its type: True (38 fields; types: {'float': 15, 'category': 9, 'int': 9, 'str': 3, 'date': 2})
every factor has the required keys: True (8 factors)

one field, as stored:
{
 "name": "annual_inc",
 "type": "float",
 "min": 1000,
 "max": 10000000,
 "nullable": false,
 "cleaning": "values below 1,000 treated as thousands (x1000); capped at p0.5 = 16,000 and p99.5 = 300,000, then log(1 + x)",
 "missing_strategy": "left missing (none in the fit window)",
 "reliability": "medium",
 "reliability_reason": "Self-reported. 899 loans (3%) typed in thousands, repaired. Consistent with credit limits and debt ratios in every verification group (rank correlation 0.55 to 0.57 with total credi

In [5]:
fields = pd.DataFrame(spec["fields"])
print(pd.crosstab(fields.role, fields.reliability).to_string())
print("\nplatform score fields (a benchmark, not a default input):", spec["platform_score_fields"])
print("restricted fields (fairness review):", spec["restricted_fields"])
print(f"\n{len(spec['excluded_fields'])} columns excluded, each with its reason, for example:")
for e in spec["excluded_fields"][:3]:
    print(f"  {e['name']} ({e['source']}): {e['reason'][:150]}")
print("\nrules the spec states for whoever scores a new application:")
for n in spec["notes"]:
    print(" -", n)

reliability     high  low  medium
role                             
excluded           0    1       0
input             21    0       9
platform_score     3    0       0
restricted         0    1       2
time_key           1    0       0

platform score fields (a benchmark, not a default input): ['int_rate', 'grade', 'sub_grade']
restricted fields (fairness review): ['emp_title', 'zip_code', 'addr_state']

12 columns excluded, each with its reason, for example:
  title (applications): Borrower-typed text; 1,821 distinct values that mostly restate purpose.
  bureau_recovery_flag (credit_bureau): Leak: identical to `recoveries > 0` in the servicing file (4,207 of 4,207 loans), and every flagged loan defaulted (single-field AUC 0.91). Always 0 i
  total_pymnt (loan_servicing): Recorded after the loan was issued; not known when the applicant applies. Payments made so far reveal the outcome. Alone it predicts default with AUC 

rules the spec states for whoever scores a new application:
 - 

**Reading it.** Every field has its type, valid range, cleaning rule, missing-value strategy, a reliability rating **with the evidence behind it** (notebook 03), its role (input, platform score, restricted, excluded) and whether it was known at application time. The engineer can decide what to trust without reading a single notebook. The spec also lists the **excluded columns with reasons**, the **WoE bins and category groupings** (notebook 04) and the **PCA and factor definitions** (notebook 05).

## 3. `transform.joblib`: one fitted object, raw in, model-ready out

In [6]:
print(type(transformer).__name__, "| contract version", transformer.contract_version)
print("steps, in order:", [name for name, _ in transformer.steps_])

ApplicationTransformer | contract version 1.0.0
steps, in order: ['cleaner', 'amounts', 'ratios', 'counts', 'features', 'feature_treater', 'categories', 'factors', 'woe']


The pipeline is one object. It takes a raw application plus its bureau record, exactly as a caller would supply it, and returns every column the model needs. Here is what it does to real rows from the reference file that contain messy values:

In [7]:
ref = read_reference(C / "reference_transforms.csv")
raw = ref[[c for c in ref.columns if not c.startswith("expected_") and c not in ("defaulted", "why_this_row")]]
out = transformer.transform(raw)

cases = [("income typed in thousands", "annual_inc_fixed", "annual_inc"), ("loan amount 10x too large", "loan_amnt_fixed", "loan_amnt"),
         ("revolving balance 10x too large", "revol_bal_fixed", "revol_bal"), ("zero coded for none on record", "mths_since_last_delinq_recoded", "mths_since_last_delinq"),
         ("placeholder interest rate", "int_rate_imputed", "int_rate")]
rows = []
for why, flag, col in cases:
    i = out.index[out[flag]][0]
    rows.append({"what was wrong": why, "field": col, "raw value": raw.loc[i, col], "after the transform": out.loc[i, col]})
i = raw.index[raw.fico_range_low > raw.fico_range_high][0]
rows.append({"what was wrong": "FICO band written backwards", "field": "fico_range_low / high",
             "raw value": f"{raw.loc[i, 'fico_range_low']:.0f} / {raw.loc[i, 'fico_range_high']:.0f}", "after the transform": f"{out.loc[i, 'fico_range_low']:.0f} / {out.loc[i, 'fico_range_high']:.0f}"})
i = raw.index[~raw.addr_state.astype(str).str.fullmatch(r"[A-Z]{2}")][0]
rows.append({"what was wrong": "state written in full or lower case", "field": "addr_state", "raw value": raw.loc[i, "addr_state"], "after the transform": out.loc[i, "addr_state"]})
print(pd.DataFrame(rows).to_string(index=False))
print(f"\n{raw.shape[1]} raw columns in, {out.shape[1]} columns out")

                     what was wrong                  field raw value after the transform
          income typed in thousands             annual_inc     114.0            114000.0
          loan amount 10x too large              loan_amnt  180000.0             18000.0
    revolving balance 10x too large              revol_bal  176600.0             17660.0
      zero coded for none on record mths_since_last_delinq       0.0                 NaN
          placeholder interest rate               int_rate       6.0               16.59
        FICO band written backwards  fico_range_low / high 699 / 695           695 / 699
state written in full or lower case             addr_state       CA                   CA

39 raw columns in, 147 columns out


### It fails fast on bad input
A model that quietly accepts a batch with a missing field or a wrong unit would produce confident nonsense. The contract refuses instead:

In [8]:
try:
    validate_raw(raw.drop(columns=["annual_inc"]))
except ContractViolation as e:
    print("a required field is missing ->", str(e)[:230])
batch = load_applications().head(500).drop(columns=["defaulted"])
try:
    transformer.transform(batch.assign(revol_util=batch.revol_util / 100))
except ValueError as e:
    print("\na batch with utilisation given as a fraction (0.91 instead of 91) ->", str(e)[:230])
print("(the check looks at the whole batch, never one row: a single 0.9 could be a real 0.9% utilisation; it needs at least 100 rows)")

a required field is missing -> input breaks the data contract:
  - missing required field 'annual_inc'

a batch with utilisation given as a fraction (0.91 instead of 91) -> ['revol_util'] look like fractions (0.91) but must be percentages (91); multiply the batch by 100 before scoring
(the check looks at the whole batch, never one row: a single 0.9 could be a real 0.9% utilisation; it needs at least 100 rows)


### It scores new applications that have no label
The 2016 holdout has no `defaulted` column and still carries the leaked recovery flag. The transform handles both:

In [9]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ho = load_holdout().head(5)
t = transformer.transform(ho)
print(f"holdout rows in: {len(ho)}; columns out: {t.shape[1]}")
print("label in the output:", "defaulted" in t.columns, "| leaked recovery flag in the output:", "bureau_recovery_flag" in t.columns)

holdout rows in: 5; columns out: 147
label in the output: False | leaked recovery flag in the output: False


**Reading it.** The same object that built the training table scores a single new application, so training and live scoring cannot disagree about how a field is cleaned. It is **fitted only on loans to the cutoff** and is never refitted: `fit` is run once, in `scripts/build_contract.py`, and the model engineer only calls `transform`.

## 4. `reference_transforms.csv` and the parity check
The reference file holds **50 raw rows chosen to exercise the pipeline**, with the expected value of every output column next to them. If an engineer's environment, or a later code change, alters any value, the parity check shows it.

In [10]:
cover = {
    "income typed in thousands": int(out.annual_inc_fixed.sum()), "loan amount 10x too large": int(out.loan_amnt_fixed.sum()),
    "revolving balance 10x too large": int(out.revol_bal_fixed.sum()), "credit date after the loan": int(out.earliest_cr_line_invalid.sum()),
    "negative bank-card utilisation": int(out.bc_util_invalid.sum()), "placeholder interest rate": int(out.int_rate_imputed.sum()),
    "zero coded for none on record": int(out.mths_since_last_delinq_recoded.sum()),
    "FICO band backwards": int((raw.fico_range_low > raw.fico_range_high).sum()), "negative revolving utilisation": int((raw.revol_util < 0).sum()),
    "state in full or lower case": int((~raw.addr_state.astype(str).str.fullmatch(r"[A-Z]{2}")).sum()),
    "date in another format": int((~raw.issue_d.astype(str).str.fullmatch(r"[A-Za-z]{3}-\d{4}")).sum()),
}
print(f"{len(ref)} reference rows, issued {pd.to_datetime(raw.issue_d, format='mixed').dt.year.value_counts().sort_index().to_dict()}")
print("rows that exercise each repair:")
print(pd.Series(cover).to_string())
print("\nthe file also records why each row is there (column why_this_row), for example:")
print(ref.why_this_row.head(4).to_string())

50 reference rows, issued {2013: 23, 2014: 8, 2015: 19}
rows that exercise each repair:
income typed in thousands          2
loan amount 10x too large          2
revolving balance 10x too large    3
credit date after the loan         2
negative bank-card utilisation     2
placeholder interest rate          2
zero coded for none on record      2
FICO band backwards                2
negative revolving utilisation     3
state in full or lower case        7
date in another format             3

the file also records why each row is there (column why_this_row), for example:
0    loan amount 10x too large
1    loan amount 10x too large
2          income in thousands
3          income in thousands


In [11]:
ok, lines = verify_contract("contract")
print("\n".join(lines))
print("\nresult:", "PASS" if ok else "FAIL")

ok   analytical_dataset.parquet
ok   transform.joblib
ok   data_spec.json
ok   reference_transforms.csv
ok   findings.md
50 reference rows, 144 expected columns compared at tolerance 1e-06; largest numeric difference 0.00e+00
parity: transform.joblib reproduces reference_transforms.csv

result: PASS


**Reading it.** The reference rows cover every repair the pipeline performs, plus ordinary rows from all three years, so they would catch a change to any of them. The check recomputes all expected columns from the raw rows with the saved transform and compares them to tolerance 1e-6 (numbers) or exactly (text, dates, flags); the largest difference is 0. The same check runs as `scripts/verify_contract.py` (exit code 0 means pass), and as a test in the model engineer's test suite.

## 5. How the contract is used downstream (the hand-off)

In [12]:
inputs = [f["name"] for f in spec["fields"] if f["role"] == "input"]
woe_cols = [c for c in ds.columns if c.endswith("_woe")]
train = ds[ds.issue_d <= FIT_CUTOFF]
valid = ds[ds.issue_d > FIT_CUTOFF]
print(f"train on {len(train):,} loans up to the cutoff, validate on {len(valid):,} loans from 2015")
print(f"{len(inputs)} fields are model inputs by role; {len(woe_cols)} WoE columns are available for the scorecard; "
      f"{ds.columns.str.startswith('pc_').sum()} PCA components and {ds.columns.str.startswith('factor_').sum()} factor scores for the ablation")
print("default rate, train vs validation:", round(train.defaulted.mean(), 3), "vs", round(valid.defaulted.mean(), 3))

train on 14,373 loans up to the cutoff, validate on 15,627 loans from 2015
30 fields are model inputs by role; 37 WoE columns are available for the scorecard; 7 PCA components and 8 factor scores for the ablation
default rate, train vs validation: 0.172 vs 0.201


The model engineer therefore: loads `analytical_dataset.parquet`; trains on loans up to the cutoff; validates on 2015; takes the WoE columns for the scorecard baseline and the raw fields, PCA components and factor scores for the ablation; reads reliability and role from `data_spec.json`; uses `transform.joblib` unchanged for scoring; and proves parity on the 50 reference rows.

## Summary: what the brief asked, and where it is answered

| Brief item | Answered in |
|---|---|
| Five contract files with the stated contents | the table at the top, and sections 1 to 4 |
| All transforms fitted only on loans up to the fit cutoff | section 3 (and `data_spec.json`: `fit_cutoff`, the notes; tests guard it) |
| `data_spec.json` in the required shape, plus reliability, bins, groupings, factors | section 2 |
| 50 reference rows and a script that verifies them at 1e-6 | section 4 |
| An engineer can model on it without questions | section 5 |
| `findings.md`: a manager summary, then the technical appendix | the file itself (reviewed separately) |